In [5]:
import glob, os
import numpy as np
import matplotlib.pyplot as plt
from pyPLUTO.pload import pload
import grid as G
import warnings
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PY_DIR, PL_DIR = 'out', '../../../'         
VARS      = ['rho', 'prs', 'vx1', 'vx2', 'vx3', 'Bx1', 'Bx2', 'Bx3', 'tr1']
LOG_VARS  = ('rho', 'prs')
RHO_TORUS = 1e-2

PLOTS_DIR = 'comparison_plots'
norm_dir  = os.path.join(PLOTS_DIR, 'norms')
maps_dir  = os.path.join(PLOTS_DIR, 'maps')
cuts_dir  = os.path.join(PLOTS_DIR, 'cuts')
for d in (norm_dir, maps_dir, cuts_dir):
    os.makedirs(d, exist_ok=True)

# Geom
f1, f2 = G.x1_faces, G.x2_faces
dV = ((f1[1:]**3 - f1[:-1]**3)/3)[:, None] * (np.cos(f2[:-1]) - np.cos(f2[1:]))[None, :]
RF, TF = np.meshgrid(f1, f2, indexing='ij')
X, Y = RF*np.sin(TF), RF*np.cos(TF)         

# Cargar paso n
def load(n):
    py = np.load(f'{PY_DIR}/data.{n:04d}.npz')
    pl = pload(n, w_dir=PL_DIR)
    return py, pl

# Compara errores
# L1 es la suma de diferencias absolutas ponderadas por el volumen
# L2 es la raiz cuadrada de diferencias al cuadrado, ponderadas
def norms(a, b, mask=None):
    w  = dV if mask is None else dV*mask
    L1 = np.sum(w*np.abs(a-b)) / max(np.sum(w*np.abs(b)), 1e-300)
    L2 = np.sqrt(np.sum(w*(a-b)**2) / max(np.sum(w*b**2), 1e-300))
    return L1, L2

# Buscar el número de salida de los archivos existentes, ordenándolos
outs = sorted(int(os.path.basename(f)[5:9]) for f in glob.glob(f'{PY_DIR}/data.*.npz'))
# Filtrado para obtener sólo parejas de archivos .npz y .dbl con el mismo n
outs = [n for n in outs if os.path.exists(f'{PL_DIR}/data.{n:04d}.dbl')]

# Mapas

def maps(py, pl, n, t, var, log=False):
    a, b = py[var], getattr(pl, var)

    fig, ax = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
    f = (lambda q: np.log10(np.maximum(q, 1e-12))) if log else (lambda q: q)
    vmin, vmax = f(b).min(), f(b).max()
    for k, (q, name) in enumerate([(a, 'Python'), (b, 'PLUTO')]):
        im = ax[k].pcolormesh(X, Y, f(q), vmin=vmin, vmax=vmax, shading='flat') 
        ax[k].set_title(f'{'log10' if log else ''}{var} {name}, t={t:.2f}')
        plt.colorbar(im, ax=ax[k])

    d = a - b
    dm = max(np.abs(d).max(), 1e-30)
    im = ax[2].pcolormesh(X, Y, d, cmap='RdBu_r', shading='flat', vmin=-dm, vmax=dm)
    ax[2].set_title('Python - PLUTO', fontsize=12); plt.colorbar(im, ax=ax[2])

    for a_ in ax:
        a_.set_aspect('equal'); a_.set_xlabel(r'$R = r \sin \theta$', fontsize=12)
    ax[0].set_ylabel(r'$z = r \cos \theta$', fontsize=12) 
    
    plt.tight_layout()
    plt.savefig(os.path.join(maps_dir, f'map_{var}_{n:04d}.png'), dpi=100)
    plt.close(fig)

# Cortes
def cuts(py, pl, n, t, var):
    b   = getattr(pl, var)
    jeq = py[var].shape[1]//2
    r   = py['x1']
    th  = py['x2']
    ir  = np.argmin(np.abs(r - 4.7))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(r, py[var][:, jeq], label='Python')
    ax1.plot(r,  b[:, jeq], '--', label='PLUTO')
    ax1.set_xlabel(r'$r$', fontsize=12); ax1.set_title(fr'{var}, $\theta=\pi /2$, t={t:.2f}')

    ax2.plot(th, py[var][ir, :], label='Python')
    ax2.plot(th,  b[ir, :], '--', label='PLUTO')
    ax2.set_xlabel(r'$\theta$', fontsize=12); ax2.set_title(fr'{var}, $r=4.7$, t={t:.2f}')

    # Escala log para variables positivas:
    if var in LOG_VARS:
        for ax in (ax1, ax2):
            ax.set_yscale('log')
        if var == 'rho':
            for ax in (ax1, ax2):
                ax.set_ylim(1e-5, 2)
    for ax in (ax1, ax2):
        ax.grid(alpha=0.3, color='gray'); ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(cuts_dir, f'cut_{var}_{n:04d}.png'), dpi=100)
    plt.close(fig)

times     = []
err_all   = {v: [] for v in VARS}   # dominio completo
err_torus = {v: [] for v in VARS}   # sólo el toro (máscara de PLUTO)

# Normas
def plot_norms(err, fname, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11,4))
    for var in VARS:
        e = np.array(err[var], dtype=float)
        e[e == 0] = np.nan
        ax1.semilogy(times, e[:, 0], 'o-', label=var)
        ax2.semilogy(times, e[:, 1], 'o-', label=var)
    ax1.set_title(f'L1 relativa ({title})', fontsize=12)
    ax2.set_title(f'L2 relativa ({title})', fontsize=12)

    for ax in (ax1, ax2):
        ax.set_xlabel(r'$t$', fontsize=12); ax.grid(alpha=0.3, color='gray')
    ax1.legend(ncol=2, fontsize=7)
    
    plt.tight_layout()
    plt.savefig(os.path.join(norm_dir, fname), dpi=100)
    plt.close(fig)

# Bucle
for n in outs:
    py, pl = load(n)
    t = pl.SimTime
    times.append(t)

    torus = (pl.rho > RHO_TORUS).astype(float)
    for var in VARS:
        b = getattr(pl, var)
        err_all[var].append(norms(py[var], b))
        err_torus[var].append(norms(py[var], b, torus))

        maps(py, pl, n, t, var, log=(var in LOG_VARS))

    cuts(py, pl, n, t, 'rho')
    print(f'Salida {n:04d} (t={t:.2f}) procesada.')

plot_norms(err_all,   'norms_all.png',   'dominio completo')
plot_norms(err_torus, 'norms_torus.png', rf'toro, $\rho>{RHO_TORUS:g}$')

Reading single data file : ../../../data.0000.dbl
Salida 0000 (t=0.00) procesada.
Reading single data file : ../../../data.0001.dbl
Salida 0001 (t=5.00) procesada.
Reading single data file : ../../../data.0002.dbl
Salida 0002 (t=10.00) procesada.
